Dataset Construction (JobHop × ESCO)

This notebook constructs the final datasets used in the thesis experiments by integrating ESCO job information with the JobHop career trajectory dataset. First, ESCO occupations_en.csv and occupationSkillRelations_en.csv are merged to create job descriptions containing occupationUri, iscoGroup, preferredLabel, description, essential_skills, and a consolidated job_text field (“Job Title. Job Description. Essential Skills”). <br>
Next, the JobHop dataset is loaded from Hugging Face (already split into train/validation/test). To prevent data leakage, all preprocessing steps are applied independently within each split. These steps include removing duplicate rows and filtering unknown/invalid occupation codes and labels, grouping experiences by person_id and retaining only individuals with at least two experiences, and constructing chronological timelines by ordering experiences by start date. For each person, the ordered occupation sequences form the CV history (cv_codes, cv_labels, and derived cv_text), while the last occupation in the sequence defines the prediction target (target_code, target_label). <br>
Finally, ESCO job descriptions are joined to the target occupation by aligning JobHop target labels with ESCO preferredLabel to handle label/version mismatches. The output is three finalized splits (train/val/test) containing: person_id, cv_codes, cv_labels, cv_text, target_code, target_label, jd_text, occupationUri, and iscoGroup, ready for negative sampling, SBERT training, and evaluation.

Rename to: 00_build_dataset_jobhop_esco.ipynb <br>
Add links to ESCO (and version) and JobHop dataset (+paper?)

<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [ ]:
import pandas as pd
import numpy as np
import os

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [31]:
occ = pd.read_csv("../Data/Job Descriptions/occupations_en.csv")
occrs = pd.read_csv("../Data/Job Descriptions/occupationSkillRelations_en.csv")
skills = pd.read_csv("../Data/Job Descriptions/skills_en.csv")

<a class="anchor" id="chapter3"></a>

# 3. Initial Analysis

</a>

<a class="anchor" id="sub-section-3_1"></a>

## 3.1. Occupations

</a>

<a class="anchor" id="sub-section-3_1_1"></a>

### 3.1.1. Types & Structure

</a>

In [32]:
print("Shape of Occupations df:", occ.shape)
occ.head()

Shape of Occupations df: (3043, 15)


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,director of technical arts\ntechnical supervis...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Technical directors realise the artistic visio...,2654.1.7,http://data.europa.eu/ux2/nace2.1/9031
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,wire drawer\nforming machine operative\ndraw m...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,http://data.europa.eu/ux2/nace2.1/242
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,precision device quality control supervisor\np...,NaN,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Precision device inspectors make sure precisio...,7543.10.3,http://data.europa.eu/ux2/nace2.1/2651
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2024-01-29T16:01:13.998Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1,http://data.europa.eu/ux2/nace2.1/5223
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,yield manager\nhospitality yields manager\nhos...,NaN,released,2024-01-11T10:28:45.871Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9,"http://data.europa.eu/ux2/nace2.1/701,\nhttp:/..."


In [33]:
occ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   conceptType              3043 non-null   object
 1   conceptUri               3043 non-null   object
 2   iscoGroup                3043 non-null   int64 
 3   preferredLabel           3043 non-null   object
 4   altLabels                3043 non-null   object
 5   hiddenLabels             8 non-null      object
 6   status                   3043 non-null   object
 7   modifiedDate             3043 non-null   object
 8   regulatedProfessionNote  3043 non-null   object
 9   scopeNote                310 non-null    object
 10  definition               8 non-null      object
 11  inScheme                 3043 non-null   object
 12  description              3043 non-null   object
 13  code                     3043 non-null   object
 14  naceCode                 3043 non-null  

<a class="anchor" id="sub-section-3_1_2"></a>

### 3.1.2. Duplicates

</a>

In [34]:
occ[occ.index.duplicated() == True]

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode


In [35]:
duplicates = occ.duplicated().sum()
print(f"There are {duplicates} duplicated rows.")

There are 0 duplicated rows.


<a class="anchor" id="sub-section-3_1_3"></a>

### 3.1.3. Missing Values

</a>

In [36]:
occ.isna().sum()

conceptType                   0
conceptUri                    0
iscoGroup                     0
preferredLabel                0
altLabels                     0
hiddenLabels               3035
status                        0
modifiedDate                  0
regulatedProfessionNote       0
scopeNote                  2733
definition                 3035
inScheme                      0
description                   0
code                          0
naceCode                      0
dtype: int64

<a class="anchor" id="sub-section-3_1_4"></a>

### 3.1.4. Statistical Analysis

</a>

In [40]:
occ.describe(include = object).T

,count,unique,top,freq
conceptType,3043,1,Occupation,3043
conceptUri,3043,3039,http://data.europa.eu/esco/occupation/4d27152a...,2
preferredLabel,3043,3039,early years teaching assistant,2
altLabels,3043,3039,reception teaching assistant\nassistant in ear...,2
hiddenLabels,8,8,sexual health consultant\nyouth policy manager...,1
status,3043,1,released,3043
modifiedDate,3043,2902,2025-06-25T07:45:00.959Z,27
regulatedProfessionNote,3043,2,http://data.europa.eu/esco/regulated-professio...,3027
scopeNote,310,274,Excludes people performing managerial activities.,12
definition,8,8,Excludes choreologist.,1


<a class="anchor" id="sub-section-3_1_5"></a>

### 3.1.5. Drop Columns

</a>

To construct the job descriptions, I will only need:
- conceptUri: ESCO occupation ID
- isoGroup: 
- preferredLabel: Job title
- description: Job description text

In [41]:
print(list(occ))

['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code', 'naceCode']


In [42]:
occ = occ.drop(['conceptType', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 
                'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'code', 
                'naceCode'], axis=1)

In [60]:
occ = occ.rename(columns={"conceptUri": "occupationUri"})

In [61]:
occ.head()

,occupationUri,iscoGroup,preferredLabel,description
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,Metal drawing machine operators set up and ope...
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,Precision device inspectors make sure precisio...
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,Air traffic safety technicians provide technic...
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,Hospitality revenue managers maximise revenue ...


<a class="anchor" id="sub-section-3_2"></a>

## 3.2. Occupation Skill Relations

</a>

<a class="anchor" id="sub-section-3_2_1"></a>

### 3.2.1. Types & Structure

</a>

In [44]:
print("Shape of Occupation Skill Relations df:", occrs.shape)
occrs.head()

Shape of Occupation Skill Relations df: (126051, 6)


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands


In [48]:
occrs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126051 entries, 0 to 126050
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   occupationUri    126051 non-null  object
 1   occupationLabel  126051 non-null  object
 2   relationType     126051 non-null  object
 3   skillType        125992 non-null  object
 4   skillUri         126051 non-null  object
 5   skillLabel       126051 non-null  object
dtypes: object(6)
memory usage: 5.8+ MB


<a class="anchor" id="sub-section-3_2_2"></a>

### 3.2.2. Duplicates

</a>

In [45]:
occrs[occrs.index.duplicated() == True]

,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel


In [46]:
duplicates = occrs.duplicated().sum()
print(f"There are {duplicates} duplicated rows.")

There are 0 duplicated rows.


<a class="anchor" id="sub-section-3_2_3"></a>

### 3.2.3. Missing Values

</a>

In [47]:
occrs.isna().sum()

occupationUri       0
occupationLabel     0
relationType        0
skillType          59
skillUri            0
skillLabel          0
dtype: int64

<a class="anchor" id="sub-section-3_2_4"></a>

### 3.2.4. Statistical Analysis

</a>

In [49]:
occrs.describe(include = object).T

,count,unique,top,freq
occupationUri,126051,3039,http://data.europa.eu/esco/occupation/9b889f07...,178
occupationLabel,126051,3039,specialised doctor,178
relationType,126051,2,essential,67600
skillType,125992,2,skill/competence,91608
skillUri,126051,13475,http://data.europa.eu/esco/skill/415abd43-e8e5...,340
skillLabel,126051,13475,use different communication channels,340


In [52]:
occrs['relationType'].unique()

array(['essential', 'optional'], dtype=object)

Filter to only essential skills

In [50]:
essential_skills = occrs[occrs['relationType'] == 'essential']

In [51]:
essential_skills['relationType'].describe()

count         67600
unique            1
top       essential
freq          67600
Name: relationType, dtype: object

Group by skill labels by occupationUri

In [ ]:
occ_to_skills = (
    essential_skills
    .groupby('occupationUri')['skillLabel']
    .apply(list)
    .reset_index(name='essential_skills')
)

In [56]:
print(occ_to_skills.shape)
occ_to_skills.head()

(3039, 2)


,occupationUri,essential_skills
0,http://data.europa.eu/esco/occupation/00030d09...,"[theatre techniques, organise rehearsals, writ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,"[quality and cycle time optimisation, types of..."
2,http://data.europa.eu/esco/occupation/0019b951...,"[quality assurance procedures, precision engin..."
3,http://data.europa.eu/esco/occupation/0022f466...,"[aircraft flight control systems, safety engin..."
4,http://data.europa.eu/esco/occupation/002da35b...,"[property management software, produce statist..."


In [57]:
occ_to_skills['essential_skills'].apply(len).describe()

count    3039.000000
mean       22.244159
std        11.991231
min         3.000000
25%        14.000000
50%        19.000000
75%        27.000000
max        99.000000
Name: essential_skills, dtype: float64

There are 3043 occupations in the Occupations dataframe, while in this dataframe there are only 3039. 4 occupations with no essential skills.

<a class="anchor" id="sub-section-3_3"></a>

## 3.3. Merge Occupations and Occupation Skill Relations

</a>

In [62]:
jd_df = (
    occ
    .merge(
        occ_to_skills,
        on='occupationUri',
        how='left'
    )
)

In [65]:
print("Shape: ", jd_df.shape)
jd_df.head()

Shape:  (3043, 5)


,occupationUri,iscoGroup,preferredLabel,description,essential_skills
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,Technical directors realise the artistic visio...,"[theatre techniques, organise rehearsals, writ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,Metal drawing machine operators set up and ope...,"[quality and cycle time optimisation, types of..."
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,Precision device inspectors make sure precisio...,"[quality assurance procedures, precision engin..."
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,Air traffic safety technicians provide technic...,"[aircraft flight control systems, safety engin..."
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,Hospitality revenue managers maximise revenue ...,"[property management software, produce statist..."


In [63]:
jd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   occupationUri     3043 non-null   object
 1   iscoGroup         3043 non-null   int64 
 2   preferredLabel    3043 non-null   object
 3   description       3043 non-null   object
 4   essential_skills  3043 non-null   object
dtypes: int64(1), object(4)
memory usage: 119.0+ KB


In [66]:
jd_df[jd_df.index.duplicated() == True]

,occupationUri,iscoGroup,preferredLabel,description,essential_skills


In [69]:
#duplicates = jd_df.duplicated().sum()
#print(f"There are {duplicates} duplicated rows.")

# Can't because essential_skills contains lists

In [70]:
jd_df['essential_skills'].apply(len).describe()

count    3043.000000
mean       22.237266
std        11.987292
min         3.000000
25%        14.000000
50%        19.000000
75%        27.000000
max        99.000000
Name: essential_skills, dtype: float64

No missing values. Duplicates? Check later.

In [ ]:
# Create job descriptions
def build_jd_text(row):
    text = f"{row['preferredLabel']}. {row['description']}"
    if row['essential_skills']:
        skills = "; ".join(row['essential_skills'])
        text += f" Essential skills: {skills}."
    return text

In [72]:
jd_df['jd_text'] = jd_df.apply(build_jd_text, axis=1)

In [73]:
jd_df['jd_text'].head()

0    technical director. Technical directors realis...
1    metal drawing machine operator. Metal drawing ...
2    precision device inspector. Precision device i...
3    air traffic safety technician. Air traffic saf...
4    hospitality revenue manager. Hospitality reven...
Name: jd_text, dtype: object

In [75]:
os.getcwd()

'/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Notebooks'

In [ ]:
#os.chdir('/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Data')

In [ ]:
#jd_df.to_csv('esco_job_descriptions_en.csv', index=False)

In [ ]:
#os.chdir('/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Notebooks')